In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


In [ ]:
# Restore conda DLL search paths before importing numerical/geometry libraries.
# VS Code can start this kernel without successfully activating its conda env.
import os
import sys
from pathlib import Path

if os.name == "nt" and (Path(sys.prefix) / "conda-meta").is_dir():
    _isovist_env = Path(sys.prefix)
    _isovist_dirs = [
        _isovist_env,
        _isovist_env / "Library" / "mingw-w64" / "bin",
        _isovist_env / "Library" / "usr" / "bin",
        _isovist_env / "Library" / "bin",
        _isovist_env / "Scripts",
        _isovist_env / "bin",
    ]
    _isovist_dirs = [str(p) for p in _isovist_dirs if p.is_dir()]
    _isovist_existing_path = os.environ.get("PATH", "").split(os.pathsep)
    _isovist_prepend = {os.path.normcase(p) for p in _isovist_dirs}
    os.environ["PATH"] = os.pathsep.join(
        _isovist_dirs + [p for p in _isovist_existing_path
                         if os.path.normcase(p) not in _isovist_prepend]
    )
    # Keep directory handles alive for the entire kernel session.
    if "_isovist_dll_handles" not in globals():
        _isovist_dll_handles = {}
    for _isovist_dir in _isovist_dirs:
        if _isovist_dir not in _isovist_dll_handles:
            _isovist_dll_handles[_isovist_dir] = os.add_dll_directory(_isovist_dir)
    print("Conda DLL paths restored for:", sys.prefix)


# Isovist Dictionaries and Metrics Tutorial

An isovist in the context of spatial analysis is the set of all points visible from a particular location in a space, typically described in terms of its geometry. In a 2D environment, an isovist is the visible area that a person can observe from a specific point, considering physical obstructions such as walls or barriers. Several metrics are commonly calculated to analyze and describe the properties of a 2D isovist. These metrics are useful in fields like architecture, urban design, and environmental psychology, as they help understand how visibility influences movement, interaction, and perception in a space.

Here are the metrics computed for a 2D isovist. The words in paranthesis is the key string used in the dictionary:

1. **Viewpoint (viewpoint)**
- **Definition**: The x, y, z coordinates of the viewpoint from which the visible space is computed.
- **Description**: This metric returns the x,y,z coordinates of the viewpoint from which the visible space is computed.
- **Calculation**: These values are extracted from the input topologic vertex.

2. **Direction (direction)**
- **Definition**: The x, y, z vector direction where the view is directed.
- **Description**: This metric returns the x, y, z vector direction of the isovist.
- **Calculation**: These values are given by the input direction vector.

3. **Field of View (fov)**
- **Definition**: The horizontal field of view (fov) angle in degrees.
- **Description**: This metric returns the horizontal field of view (fov) angle in degrees.
- **Calculation**: This value is given by the input fov parameter.

4. **Area (area)**
- **Definition**: The area of the isovist is the total size of the visible space from a specific location.
- **Description**: This metric calculates how much space can be seen from a given point, which can be important in understanding how open or enclosed a space feels. A larger isovist area typically implies a more open environment, while a smaller area suggests a more confined or obstructed space.
- **Calculation**: It is calculated as the total area enclosed by the boundary of the isovist polygon.

5. **Perimeter (perimeter)**
- **Definition**: The perimeter of an isovist is the length of the boundary or edge of the visible area.
- **Description**: This metric helps to quantify the complexity or length of the visible boundary. A large, irregular perimeter indicates a more complex boundary with potentially more visual information, while a smooth, simple perimeter might imply fewer visual cues.
- **Calculation**: The perimeter is calculated by summing the lengths of all the edges of the isovist's boundary.

6. **Compactness (compactness)**
- **Definition**: Compactness measures how closely the shape of the isovist approximates a circle (the most compact geometric shape).
- **Description**: A more compact isovist suggests a more symmetric and less fragmented visual field, while a less compact isovist indicates a more elongated or irregular shape. This metric can be important in understanding how easily a person can visually navigate the space.
- **Calculation**: It is typically computed using the formula:
     \[
     C = \frac{P^2}{4\pi A}
     \]
     where \( P \) is the perimeter and \( A \) is the area. The closer the value is to 1, the more compact the shape. A value greater than 1 indicates an irregular shape.

7. **Maximum Visibility Distance / Vista Length (d_max)**
- **Definition**: The maximum visibility distance / vista length is the longest straight line that can be seen from the viewpoint.
- **Description**: This metric reflects how far one can see in the space from a given point. It can give an idea of how open or obstructed a space feels in terms of sightlines.
- **Calculation**: It is determined by calculating the longest straight line from the observation point to the outer borders of the isovist.

8. **Minimum Visibility Distance (d_min)**
- **Definition**: The minimum visibility distance is the shortest straight line that can be seen from the viewpoint.
- **Description**: This metric reflects how far one can see in the space from a given point. It can give an idea of how open or obstructed a space feels in terms of sightlines.
- **Calculation**: It is determined by calculating the shortest straight line from the observation point to the outer borders of the isovist.

9. **Average Visibility Distance (d_avg)**
- **Definition**: The average visibility distance is the average of all straight lines that can be seen from the viewpoint.
- **Description**: This metric reflects the average of how much one can see in the space from a given point. It can give an idea of how open or obstructed a space feels in terms of sightlines.
- **Calculation**: It is determined by calculating the average straight line from the observation point to the outer borders of the isovist.

10. **Furthest Vertex (v_max)**
- **Definition**: The x, y, z coordinates of the furthest vertex that can be seen from the viewpoint without encountering an obstacle.
- **Description**: This metric reflects how far one can see in the space from a given point. It can give an idea of how open or obstructed a space feels in terms of sightlines.
- **Calculation**: It is determined by calculating the coordinates of the vertex with the largest distance from the viewpoint that does not intersect with any boundary or obstruction (e.g., walls or barriers).

11. **Closest Vertex (v_min)**
- **Definition**: The x, y, z coordinates of the furthest vertex that can be seen from the viewpoint without encountering an obstacle.
- **Description**: This metric reflects how close one can see in the space from a given point. It can give an idea of how open or obstructed a space feels in terms of sightlines.
- **Calculation**: It is determined by calculating the coordinates of the vertex with the smallest distance from the viewpoint that does not intersect with any boundary or obstruction (e.g., walls or barriers).

12. **Centroid (centroid)**
- **Definition**: The x, y, z coordinates of the centroid of the isovist.
- **Description**: This metric reflects the position of the center of the visible space.
- **Calculation**: It is determined by calculating average of the coordinates of the vertices of the isovist polygon.

13. **Visibility Distribution (v_d)**
- **Definition**: Visibility distribution quantifies the distribution of visible points across the isovist.
- **Description**: This metric can tell you whether the visibility from a point is more spread out or concentrated in a certain direction. A uniform visibility distribution indicates a more balanced visual field, while a skewed distribution suggests that the observer's line of sight is constrained in certain directions.
- **Calculation**: It is calculated by measuring the angular distribution of visible points or by computing the density of visible locations within sectors of the isovist.

14. **Viewpoint Density (v_density)**
- **Definition**: Viewpoint density refers to the number of visible points per unit area within the isovist.
- **Description**: A higher viewpoint density means that the space is more densely populated with visible points, which could indicate a higher degree of visual richness or potential for interaction in that space.
- **Calculation**: It is calculated by dividing the total number of visible points by the area of the isovist:
     \[
     VDensity = \frac{\text{Number of visible points}}{A}
     \]
     where \( A \) is the area of the isovist.

15. **Isovist Symmetry (symmetry)**
- **Definition**: Symmetry measures how balanced or symmetrical the isovist is around the point of observation.
- **Description**: A high symmetry value indicates a balanced visual field in all directions, while low symmetry indicates that the space is more irregular or skewed. Symmetry can influence how a person perceives the space.
- **Calculation**: Symmetry is often calculated using geometric analysis of the isovist's shape. One method is to compute the average deviation from a central symmetry axis or the deviation from a perfect circular form.

16. **Fractal Dimension (d_f)**
- **Definition**: The fractal dimension measures the complexity of the isovist's boundary.
- **Description**: This metric gives insight into how rough or intricate the boundary of the visible area is. A higher fractal dimension typically indicates a more complex and jagged boundary, while a lower fractal dimension suggests a simpler, more regular boundary.
- **Calculation**: The fractal dimension is calculated using methods like box-counting or other fractal analysis techniques, which measure how the boundary detail changes with different scales of observation.

17. **Edge Complexity (e_c)**
- **Definition**: Edge complexity measures how complex the edges of the isovist boundary are.
- **Description**: More complex edges might suggest that the observer's view is influenced by more intricate shapes or features, such as angled walls or multiple obstacles.
- **Calculation**: It is calculated by analyzing the number of corners or the angle changes in the boundary of the isovist. More corners or abrupt changes in direction would indicate higher edge complexity.

18. **Mean Visual Field Angle (theta)**
- **Definition**: This metric measures the average angular extent of the visible area from the observation point.
- **Description**: The mean visual field angle gives a sense of how wide the visual range is around a person, influencing how much of the environment they can see in a given direction.
- **Calculation**: It is calculated by averaging the angular distances between visible boundary points in the isovist. This is typically expressed in degrees.

19. **Occlusivity (occlusivity)**
- **Definition**: This metric measures the proportion of edges of an isovist that are not physically defined.
- **Description**: "Occlusivity fields show moments of dramatic visual change as a user passes between spaces. A forest or hypostyle hall has high Occlusivity; a convex room has none; flat surfaces tend to extend occlusive edges into space." (from isovists.org)
- **Calculation**: It is calculated by computing the total length of all occlusive edges of the isovist and dividing that length by the length of the perimeter of the isovist. This yields a relative value ranging from 0 to 1.

20. **Drift (drift)**
- **Definition**: This metric measures the distance from the observation point to the centroid of its isovist.
- **Description**: "In an isovist field, drift identifies the inherent ‘flow’ within a series of spaces, or the ‘pull’ or ‘push’ one might feel from the volume of space itself. To move along contours low in Drift is to be in the centre of one’s isovist, and so be visible from all directions. High Drift identifies regions from which space can be surveyed with a minimum of head turning." (from isovists.org)
- **Calculation**: It is calculated by computing the distance from the observation point to the centroid of the isovst polygon.

21. **Closed Perimeter (closed_perimeter)**
- **Definition**: This metric measures "the length of the closed edges (i.e. solid walls) of all space visible from a location. In isovist terminology, it represents the non occluded perimeter at said location (Benedikt, 1979)." (from isovists.org)
- **Description**: see definition.
- **Calculation**: It is calculated by computing the total length of all occlusive edges of the polygon of the isovist then subtracting that sum from the length of the perimeter of the isovist.

22. **Average Radial (average_radial)**
- **Definition**: This metric measures "the mean view length of all space visible from a location." (from isovists.org)
- **Description**: "The average radial relates to the ‘through vision’ at a location (Turner, 2007). In isovist terminology, it represents the mean radial length at a location." (from isovists.org)
- **Calculation**: It is calculated by computing the average distance between each edge of the polygon of the isovist, and the observaion point of the isovist. It then calculates the average of all such values, as weighted by the angle subtended by each edge. (from isovists.org)

23. **Variance (variance)**
- **Definition**: This metric measures "the mean of the square of deviation between all radial lengths and average radial length of an isovist (Benedikt, 1979)" (from isovists.org)
- **Description**: "In visibility graph terminology, it represents the point second moment for a location. Variance indicates both the complexity and eccentricity of an isovist. It takes other measures, studied simultaneously, to sort out which. For example, low Drift and high Variance will indicate complexity but not eccentricity (i.e. non centered-ness in one’s isovist)." (from isovists.org)
- **Calculation**: It is calculated by computing the average distance between each edge of the polygon of the isovist, and the origin point of the isovist calculated at V. It then finds the square of the difference between the latter and Average Radial (Qv). It subsequently calculates the mean of all such values, as weighted by the arc angle associated to each edge. The final outcome is square rooted. (from isovists.org)

24. **Skewness (skewness)**
- **Definition**: the mean of the cube of deviation between all radial lengths and average radial length of an isovist (Benediky, 1979)." (from isovists.org)
- **Description**: "In visibility graph terminology, it represents the point third moment for a location. When Skewness is positive, long radials are few in number and short radials many; when it is negative, the opposite is the case. Cave interiors have positive Skewness; space near columns, trees or outside corners have negative Skewness. The latter might correlate with a sense of exposure, the former with safety or power." (from isovists.org)
- **Calculation**: It is calculated by computing the average distance between each edge of the polygon of the isovist, and the origin point of the isovist calculated at V. It then finds the cube of the difference between the latter and Average Radial (Qv). It subsequently calculates the mean of all such values, as weighted by the arc angle associated to each edge. The final outcome is cube rooted. (from isovists.org)


In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

from topologicpy.Helper import Helper
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Topology import Topology
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Wire import Wire
from topologicpy.TGraph import TGraph #importing the Graph function from Topologic
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color

# Get the named colors to use
colors = Color.CSSNamedColors()

# Create the overall boundary. A 10x10 Rectangle.
r = Face.Rectangle(width=10, length=10)

# Get the edges of the boundary to assign them different dictionaries and colors
edges = Topology.Edges(_as_topology(r))
for i, edge in enumerate(edges):
    d = Dictionary.ByKeysValues(["type","edgeColor", "edgeWidth", "faceColor"], ["boundary",colors[i+30], 5, colors[i+30]])
    edge = Topology.SetDictionary(edge, d)

# Create two obstacles
r1 =Wire.Rectangle(width=2, length=2)
r1 = Topology.Translate(_as_topology(r1), x=3, y=2.5, z=0)
r2 =Wire.Rectangle(width=4, length=2)
r2 = Topology.Translate(_as_topology(r2), x=2, y=-2, z=0)

# Get the edges of the first obstacle to assign them different types and colors
edges = Topology.Edges(_as_topology(r1))
for i, edge in enumerate(edges):
    d = Dictionary.ByKeysValues(["type", "edgeColor", "edgeWidth","faceColor"], ["retail","red", 5, "red"])
    edge = Topology.SetDictionary(edge, d)

# Get the edges of the second obstacle to assign them different dictionaries and colors
edges = Topology.Edges(_as_topology(r2))
for i, edge in enumerate(edges):
    d = Dictionary.ByKeysValues(["type","edgeColor", "edgeWidth", "faceColor"], ["residential","yellow", 5, "yellow"])
    edge = Topology.SetDictionary(edge, d)

# Create the Isovist viewpoint
v = Vertex.ByCoordinates(-1,3,0)
vertex_d = Dictionary.ByKeyValue("vertexSize", 10)
v = Topology.SetDictionary(v, vertex_d)

# Calculate the Isovist
isovist = Face.Isovist(face=r,
                       obstacles=[r1, r2],
                       vertex=v,
                       fov=140,
                       direction=[1,-1,0],
                       transferDictionaries=True,
                       metrics=True,
                       triangles=True)

if isovist:
    isovist_d = _record_dictionary(isovist)
    # Print the keys and values of the resulting isovist
    keys = Dictionary.Keys(isovist_d)
    for i, key in enumerate(keys):
        print(str(i+1).zfill(2)+".", key, Dictionary.ValueAtKey(isovist_d, key))

    # VISUALIZATION
    # Set the edge width of the isovist to 5 pixels
    edges = Topology.Edges(_as_topology(isovist))
    for edge in edges:
        edge_d = _record_dictionary(edge)
        edge_d = Dictionary.SetValueAtKey(edge_d, "edgeWidth", 1)
        edge = Topology.SetDictionary(edge, edge_d)

    # Make a copy of the Isovist and move it by X=12
    isovist2 = Topology.Translate(_as_topology(isovist), 12, 0, 0)

    # Make two copies of the viewpoint and move them by X=12 and X=-12
    v_a = Topology.Translate(_as_topology(v), 12, 0, 0)
    v_b = Topology.Move(v, -12, 0, 0)

    # Make one copy of the boundary and obstacles and move them by X=-12 
    r_b = Topology.Move(r, -12, 0, 0)
    r1_b = Topology.Move(r1, -12, 0, 0)
    r2_b = Topology.Move(r2, -12, 0, 0)

    # Calculate the furthest point and construct an edge, give it a green color and a width of 3.
    v_furthest = Vertex.ByCoordinates(_as_topology(Dictionary.ValueAtKey(isovist_d, 'v_max')))
    max_edge = Edge.ByVertices(_as_topology(v), _as_topology(v_furthest))
    edge_d = Dictionary.ByKeysValues(["edgeColor", "edgeWidth"], ["green", 3])
    max_edge = Topology.SetDictionary(max_edge, edge_d)

    # Calculate the closest point and construct an edge, give it an purple color and a width of 3.
    v_closest = Vertex.ByCoordinates(_as_topology(Dictionary.ValueAtKey(isovist_d, 'v_min')))
    min_edge = Edge.ByVertices(_as_topology(v), _as_topology(v_closest))
    edge_d = Dictionary.ByKeysValues(["edgeColor", "edgeWidth"], ["purple", 3])
    min_edge = Topology.SetDictionary(min_edge, edge_d)

    # Add a faceOpacity key and value to the Isovist Face dictionary and set it to the Isovist face
    isovist_d = Dictionary.SetValueAtKey(isovist_d, "faceOpacity", 0.5)
    isovist = Topology.SetDictionary(isovist, isovist_d)

    # Show the result
    Topology.Show(r, r1,r2,r_b,r1_b,r2_b,isovist,isovist2,v,v_a, v_b, max_edge, min_edge, vertexSizeKey="vertexSize", edgeColor="black",
                edgeColorKey="edgeColor", edgeWidth=4, edgeWidthKey="edgeWidth", faceOpacityKey="faceOpacity", showFaces=True, backgroundColor="grey", camera=[0,0,3])

In [ ]:
# Get the color-coded subtended triangles
triangles = Topology.Contents(isovist)
for triangle in triangles:
    d = _record_dictionary(triangle)
    d = Dictionary.SetValueAtKey(d, "edgeWidth", 0.001)
    triangle = Topology.SetDictionary(triangle, d)
    edges = Topology.Edges(_as_topology(triangle))
    for edge in edges:
        d = _record_dictionary(edge)
        d = Dictionary.SetValueAtKey(d, "edgeWidth", 0.001)
        edge = Topology.SetDictionary(edge, d)
Topology.Show(r, r1, r2, triangles,v, vertexSizeKey="vertexSize", edgeColor="black",
                edgeColorKey="edgeColor", edgeWidth=1, edgeWidthKey="edgeWidth",
                faceOpacityKey="faceOpacity", faceColorKey="faceColor",
                showFaces=True, backgroundColor="white", camera=[0,0,3])
